<a href="https://colab.research.google.com/github/MOHAMM12342/n8n-render/blob/main/nb/Meta_Synthetic_Data_Llama3_2_(3B).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!


<a href="https://github.com/meta-llama/synthetic-data-kit"><img src="https://raw.githubusercontent.com/unslothai/notebooks/refs/heads/main/assets/meta%20round%20logo.png" width="137"></a>
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News


Train MoEs - DeepSeek, GLM, Qwen and gpt-oss faster with 32% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

You can now train embedding models 1.8-3.3x faster with 20% less VRAM. [Blog](https://unsloth.ai/docs/new/embedding-finetuning)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

3x faster LLM training with 30% less VRAM and 500K context. [3x faster](https://unsloth.ai/docs/new/3x-faster-training-packing) • [500K Context](https://unsloth.ai/docs/new/500k-context-length-fine-tuning)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm synthetic-data-kit==0.0.3
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.10.2', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install synthetic-data-kit==0.0.3
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [2]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.10.2', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### Unsloth

## Primary Goal
Our goal is to make Llama 3.2 3B understand the "Byte Latent Transformer: Patches Scale Better Than Tokens" [research paper](https://ai.meta.com/research/publications/byte-latent-transformer-patches-scale-better-than-tokens/) that was published in December 2024.

We'll use https://github.com/meta-llama/synthetic-data-kit to generate question and answer pairs **fully locally** which will be used for finetuning Llama 3.2 3B!

In [74]:
from unsloth.dataprep import SyntheticDataKit

generator = SyntheticDataKit.from_pretrained(
    # Choose any model from https://huggingface.co/unsloth
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = 2048, # Longer sequence lengths will be slower!
)

INFO 02-05 21:51:20 [vllm_utils.py:702] Unsloth: Patching vLLM v1 graph capture
INFO 02-05 21:51:20 [vllm_utils.py:731] Unsloth: Patching vLLM v0 graph capture
Unsloth: Using dtype = torch.float16 for vLLM.
Unsloth: vLLM loading unsloth/Llama-3.2-3B-Instruct with actual GPU utilization = 72.29%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.74 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 32.
Unsloth: vLLM's KV Cache can use up to 4.67 GB. Also swap space = 0 GB.
Unsloth: `cudagraph_mode` is not in `from vllm.config import CompilationConfig`
Unsloth: Disabling `disable_cascade_attn` in vLLM to allow for better on policy RL!
vLLM STDOUT: INFO 02-05 21:51:36 [__init__.py:244] Automatically detected platform cuda.
vLLM STDOUT: INFO 02-05 21:51:39 [api_server.py:1395] vLLM API server version 0.9.2
vLLM STDOUT: INFO 02-05 21:51:39 [cli_args.py:325] non-default args: {'model': 'unsloth/Llama-3.2-3B-Instruct', 'dtype': 'float16', '

## Generate QA Pairs + Auto clean data
We now use synthetic data kit for question answer pair generation:

In [4]:
generator.prepare_qa_generation(
    output_folder = "data", # Output location of synthetic data
    temperature = 0.7, # Higher temp makes more diverse datasets
    top_p = 0.95,
    overlap = 64, # Overlap portion during chunking
    max_generation_tokens = 512, # Can increase for longer QA pairs
)

Check if it succeeded:

In [25]:
!synthetic-data-kit system-check

vLLM STDOUT: INFO:     127.0.0.1:44896 - "GET /v1/models HTTP/1.1" 200 OK
 VLLM server is running at http://localhost:8000/v1
Available models: {'object': 'list', 'data': [{'id': 
'unsloth/Llama-3.2-3B-Instruct', 'object': 'model', 'created': 1770322573, 
'owned_by': 'vllm', 'root': 'unsloth/Llama-3.2-3B-Instruct', 'parent': None, 
'max_model_len': 2048, 'permission': [{'id': 
'modelperm-b8c8a5c6ea6647c8a2233da980838e80', 'object': 'model_permission', 
'created': 1770322573, 'allow_create_engine': False, 'allow_sampling': True, 
'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 
'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': 
False}]}]}
⠋ Checking VLLM server at http://localhost:8000/v1...


## Document Parsing (PDF, CSV, HTML etc.)
Now, let's take the Byte Latent Transformer: Patches Scale Better Than Tokens research paper found at https://arxiv.org/abs/2412.09871 and covert it to Q&A pairs in order to finetune Llama 3.2!

In [71]:
# Byte Latent Transformer: Patches Scale Better Than Tokens paper in HTML format
!pip install arabic-reshaper python-bidi
import requests
import pdfplumber
import os
import arabic_reshaper
from bidi.algorithm import get_display

# --- FONCTION DE RÉPARATION ---
def fix_arabic_text(text):
    if not text:
        return ""

    # 1. On "reshape" les lettres (pour qu'elles s'attachent : initial, medial, final)
    reshaped_text = arabic_reshaper.reshape(text)

    # 2. On applique l'algorithme BiDi (pour remettre de droite à gauche)
    bidi_text = get_display(reshaped_text)

    return bidi_text

# --- VOTRE SCRIPT ADAPTÉ ---

pdf_url = "https://center-family.ma/sites/default/files/publications/documents/%D8%A7%D9%84%D8%AF%D9%8A%D9%84%20%D8%A7%D9%84%D9%85%D9%8A%D8%B3%D8%B1%20%D9%84%D8%A5%D8%AC%D8%B1%D8%A7%D8%A1%D8%A7%D8%AA%20%D8%A3%D9%82%D8%B3%D8%A7%D9%85%20%D9%82%D8%B6%D8%A7%D8%A1%20%D8%A7%D9%84%D8%A3%D8%B3%D8%B1%D8%A9.pdf"
temp_pdf_path = "temp_document.pdf"
output_txt_path = "data/output/center-family_ma.txt"

os.makedirs(os.path.dirname(output_txt_path), exist_ok=True)

# 1. Téléchargement
if not os.path.exists(temp_pdf_path):
    print("Téléchargement du PDF...")
    response = requests.get(pdf_url)
    with open(temp_pdf_path, 'wb') as f:
        f.write(response.content)

# 2. Extraction et Correction
print("Extraction et correction de l'arabe en cours...")
full_text = ""

try:
    with pdfplumber.open(temp_pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            # Extraction brute
            raw_text = page.extract_text()

            if raw_text:
                # CORRECTION ICI : On traite chaque ligne pour la remettre à l'endroit
                lines = raw_text.split('\n')
                fixed_lines = [fix_arabic_text(line) for line in lines]
                page_content = "\n".join(fixed_lines)

                full_text += page_content + "\n\n"

            if i % 5 == 0:
                print(f"Page {i} traitée...")

except Exception as e:
    print(f"Erreur : {e}")

# 3. Sauvegarde
with open(output_txt_path, "w", encoding="utf-8") as f:
    f.write(full_text)

print(f"\nTerminé ! Vérifiez le fichier : {output_txt_path}")

# Petit test d'affichage pour voir si c'est lisible maintenant
print("-" * 30)
print("Aperçu des 300 premiers caractères corrigés :")
print(full_text[:300])
print("-" * 30)# Truncate document
filenames = generator.chunk_data("data/output/center-family_ma.txt")
print(len(filenames), filenames[:3])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 16.9 MB/s eta 0:00:00
Téléchargement du PDF...
Extraction et correction de l'arabe en cours...
Page 0 traitée...
Page 5 traitée...
Page 10 traitée...
Page 15 traitée...
Page 20 traitée...
Page 25 traitée...
Page 30 traitée...
Page 35 traitée...
Page 40 traitée...
Page 45 traitée...
Page 50 traitée...
Page 55 traitée...
Page 60 traitée...
Page 65 traitée...
Page 70 traitée...
Page 75 traitée...
Page 80 traitée...
Page 85 traitée...
Page 90 traitée...
Page 95 traitée...
Page 100 traitée...
Page 105 traitée...
Page 110 traitée...
Page 115 traitée...
Page 120 traitée...
Page 125 traitée...
Page 130 traitée...
Page 135 traitée...
Page 140 traitée...
Page 145 traitée...
Page 150 traitée...
Page 155 traitée...
Page 160 traitée...
Page 165 traitée...
Page 170 traitée...
Page 175 traitée...

Terminé ! Vérifiez le fichier : data/output/center-family_ma.txt
------------------------------
Aperçu des 300 premiers caractères corrigés :
ﻼﻧﺍ�ﺮ

We see around 37 chunks of data. We now call synthetic-data-kit to create some pairs of data for 3 of our chunks.

You can process more chunks, but it'll be much slower!

Using `--num-pairs` will generate **approximately** that many QA pairs. However it might be shorter or longer depending on the `max_seq_length` of the loaded up model. So if you specify 100, you might only get 10 since the model's max sequence length is capped.

In [75]:
import time
# Process 3 chunks for now -> can increase but slower!
for filename in filenames[:3]:
    !synthetic-data-kit \
        -c synthetic_data_kit_config.yaml \
        create {filename} \
        --num-pairs 25 \
        --type "qa"
    time.sleep(2) # Sleep some time to leave some room for processing

vLLM STDOUT: INFO:     127.0.0.1:40008 - "GET /v1/models HTTP/1.1" 200 OK
vLLM STDOUT: INFO:     127.0.0.1:40012 - "GET /v1/models HTTP/1.1" 200 OK
⠙ Generating qa content from data/output/center-family_ma_0.txt...vLLM STDOUT: INFO 02-05 22:01:47 [chat_utils.py:444] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
⠹ Generating qa content from data/output/center-family_ma_0.txt...vLLM STDOUT: INFO 02-05 22:01:47 [logger.py:43] Received request chatcmpl-89fea32d7dd04081a0ab1d95c60c5add: prompt: '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 05 Feb 2026\n\nSummarize this document in 3-5 sentences, focusing on the main topic and key concepts.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nﻼﻧﺍ�ﺮﺳ : ﻡﺮﻛﺯ ﻻﺩﺭﺍ�ﺱﺎﺗ ﺍ ﻷ�ﺮﺴﻳﺓ ﻭﻼﺒﺤﺛ ﻲﻓ ﻼﻘﻴﻣ ﻭﻼﻗﺎﻧﻮﻧ\nﺎﻤﻟ ﺅﻮﻠﻓ : ﻻﺪﻜﺗﻭﺭ ﺢﻤﻣﺩ ﺍﺈﻜﻴﺟ\nﻼﻌﻧﻭﺎﻧ : ﻻﺪﻠﻴﻟ ﺎﻤﻠﻳ�ﺮﺳ ﻺﺟﺭﺍﺀﺎﺗ ﺍﺄﻗ�ﺱﺎﻣ ﻕ�ﺱﺍﺀ ﺍ ﻷ�ﺮﺳﺓ\n�ﺮﺴﺣ ﻊﻤﻠﻳ ﻢﻟﺩﻮﻧﺓ ﺍ ﻷ�

Optionally, you can clean up the data via pruning "bad" or low quality examples and convert the rest to JSON format for finetuning!

In [17]:
# !synthetic-data-kit \
#     -c synthetic_data_kit_config.yaml \
#     curate --threshold 0.0 \
#     "data/generated/arxiv_org_0_qa_pairs.json"

We now convert the generated datasets into QA formats so we can load it for finetuning:

In [29]:
qa_pairs_filenames = [
    f"data/generated/center-family_ma_{i}_qa_pairs.json"
    for i in range(len(filenames[:3]))
]
for filename in qa_pairs_filenames:
    !synthetic-data-kit \
        -c synthetic_data_kit_config.yaml \
        save-as {filename} -f ft

⠋ Converting data/generated/center-family_ma_0_qa_pairs.json to ft format with 
json storage...
 Converted to ft format and saved to 
data/final/center-family_ma_0_qa_pairs_ft.json
⠋ Converting data/generated/center-family_ma_1_qa_pairs.json to ft format with 
json storage...
 Converted to ft format and saved to 
data/final/center-family_ma_1_qa_pairs_ft.json
⠋ Converting data/generated/center-family_ma_2_qa_pairs.json to ft format with 
json storage...
 Converted to ft format and saved to 
data/final/center-family_ma_2_qa_pairs_ft.json


Let's load up the data and see what the synthetic data looks like!

In [30]:
from datasets import Dataset
import pandas as pd
final_filenames = [
    f"data/final/center-family_ma_{i}_qa_pairs_ft.json"
    for i in range(len(filenames[:3]))
]
conversations = pd.concat([
    pd.read_json(name) for name in final_filenames
]).reset_index(drop = True)

dataset = Dataset.from_pandas(conversations)

In [67]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
dataset.to_pandas()

,messages,text
0,"[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'What is the document type?', 'role': 'user'}, {'content': 'PDF', 'role': 'assistant'}]",<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 05 Feb 2026\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the document type?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nPDF<|eot_id|>
1,"[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'What is the first object in the PDF?', 'role': 'user'}, {'content': 'Page object', 'role': 'assistant'}]",<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 05 Feb 2026\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the first object in the PDF?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nPage object<|eot_id|>
2,"[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'What is the purpose of the page object?', 'role': 'user'}, {'content': 'To define the page layout and content', 'role': 'assistant'}]",<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 05 Feb 2026\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the purpose of the page object?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nTo define the page layout and content<|eot_id|>
3,"[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'What is the 0th object in the PDF?', 'role': 'user'}, {'content': 'Font object', 'role': 'assistant'}]",<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 05 Feb 2026\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the 0th object in the PDF?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nFont object<|eot_id|>
4,"[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'What is the purpose of the font object?', 'role': 'user'}, {'content': 'To define the font and its properties', 'role': 'assistant'}]",<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 05 Feb 2026\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the purpose of the font object?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nTo define the font and its properties<|eot_id|>
5,"[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'What is the 5th object in the PDF?', 'role': 'user'}, {'content': 'Stream object', 'role': 'assistant'}]",<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 05 Feb 2026\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the 5th object in the PDF?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nStream object<|eot_id|>
6,"[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'What is the purpose of the stream object?', 'role': 'user'}, {'content': 'To define the binary data that makes up the PDF', 'role': 'assistant'}]",<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 05 Feb 2026\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the purpose of the stream object?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nTo define the binary data that makes up the PDF<|eot_id|>
7,"[{'content': 'You are a helpful assistant.', 'role': 'system'}, {'content': 'What is the 12th object in the PDF?', 'role': 'user'}, {'content': 'ProcSet object', 'role': 'assistant'}]",<|begin_of_text|><|start_header_id|>syste

In [32]:
dataset[1]

{'messages': [{'content': 'You are a helpful assistant.', 'role': 'system'},
  {'content': 'What is the first object in the PDF?', 'role': 'user'},
  {'content': 'Page object', 'role': 'assistant'}]}

In [33]:
dataset[-1]

{'messages': [{'content': 'You are a helpful assistant.', 'role': 'system'},
  {'content': 'Is the PDF document a binary file?', 'role': 'user'},
  {'content': 'Yes, the PDF document is a binary file.', 'role': 'assistant'}]}

Finally free vLLM process to save memory and to allow for finetuning!

In [34]:
generator.cleanup()

Attempting to terminate the VLLM server gracefully...
vLLM STDOUT: INFO 02-05 20:29:24 [launcher.py:80] Shutting down FastAPI HTTP server.
Server terminated gracefully.


### Fine-tuning Synthetic Dataset with Unsloth

In [35]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",
    # Qwen3 new models
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    # Other very popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [36]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.1.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the `Llama-3.2` format for conversation style finetunes. The chat template renders conversations like below: (Cutting Knowledge Date is by default there!)

```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 01 May 2025

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is 1+1?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

2<|eot_id|>
```

In [37]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

# Get our previous dataset and format it:
dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

See result of the first row:

In [38]:
dataset[0]

{'messages': [{'content': 'You are a helpful assistant.', 'role': 'system'},
  {'content': 'What is the document type?', 'role': 'user'},
  {'content': 'PDF', 'role': 'assistant'}],
 'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 05 Feb 2026\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the document type?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nPDF<|eot_id|>'}

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [39]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/49 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [40]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49 | Num Epochs = 9 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,4.881600
2,5.003600
3,4.684400
4,4.295300
5,3.726400
6,3.414300
7,2.519900
8,2.746000
9,1.966200
10,1.865100


In [50]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

NameError: name 'start_gpu_memory' is not defined

<a name="Inference"></a>
### Inference
Let's run the model! Use `apply_chat_template` with `add_generation_prompt` set to `True` for inference.

In [55]:
messages = [
    {"role": "user", "content": " قانون الأسرة"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer,
                   max_new_tokens = 256, temperature = 0.1)

قانون الأسرة هو قانون يحدد العلاقة بين الأفراد في الأسرة، ويتضمن العلاقة بين الأب والابن، والابنة والابن، والزوج والزوجة، وغيرها من العلاقات في الأسرة.<|eot_id|>


The model learns about the research paper!

In [60]:
messages = [
    {"role": "user", "content": " ما هو مصدر معلوماتك الاسم" },
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer,
                   max_new_tokens = 256, temperature = 0.1)

PDF<|eot_id|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("meta_synthetic_data_lora")  # Local saving
tokenizer.save_pretrained("meta_synthetic_data_lora")
# model.push_to_hub("your_name/meta_synthetic_data_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/meta_synthetic_data_lora", token = "YOUR_HF_TOKEN") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "meta_synthetic_data_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
messages = [
    {"role": "user", "content": "What is so special about BLT's tokenization?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer,
                   max_new_tokens = 256, temperature = 0.1)

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("meta_synthetic_data_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: # Change to True to upload finetune
    model.push_to_hub_merged("HF_USERNAME/meta_synthetic_data_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False:
    model.save_pretrained_merged("meta_synthetic_data_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: # Change to True to upload finetune
    model.push_to_hub_merged("HF_USERNAME/meta_synthetic_data_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("meta_synthetic_data_lora")
    tokenizer.save_pretrained("meta_synthetic_data_lora")
if False: # Change to True to upload finetune
    model.push_to_hub("HF_USERNAME/meta_synthetic_data_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/meta_synthetic_data_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

In [ ]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf("meta_synthetic_data_finetune", tokenizer,)
if False: # Change to True to upload finetune
    model.push_to_hub_gguf("HF_USERNAME/meta_synthetic_data_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("meta_synthetic_data_finetune", tokenizer, quantization_method = "f16")
if False: # Change to True to upload finetune
    model.push_to_hub_gguf("HF_USERNAME/meta_synthetic_data_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("meta_synthetic_data_finetune", tokenizer, quantization_method = "q4_k_m")
if False: # Change to True to upload finetune
    model.push_to_hub_gguf("HF_USERNAME/meta_synthetic_data_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>